# Data cleaning

In [1]:
import pandas as pd
import numpy as np
from configurations import TEST_PATH

train_df = pd.read_csv('train_folds.csv')
test_df = pd.read_csv(TEST_PATH)

print("Hiányzó értékek a train-ben:")
print(train_df[['Name', 'SexuponOutcome', 'AgeuponOutcome']].isnull().sum())

print("\nHiányzó értékek a test-ben:")
print(test_df[['Name', 'SexuponOutcome', 'AgeuponOutcome']].isnull().sum())

Hiányzó értékek a train-ben:
Name              7691
SexuponOutcome       1
AgeuponOutcome      18
dtype: int64

Hiányzó értékek a test-ben:
Name              3225
SexuponOutcome       0
AgeuponOutcome       6
dtype: int64


In [3]:
def parse_age_in_months(age_str):
    if pd.isna(age_str):
        return np.nan
    parts = str(age_str).split()
    if len(parts) < 2:
        return np.nan

    number = float(parts[0])
    unit = parts[1].lower()

    if 'year' in unit:
        return number * 12.0
    elif 'month' in unit:
        return number * 1.0
    elif 'week' in unit:
        return number / 4.33
    elif 'day' in unit:
        return number / 30.0
    return np.nan

train_df['AgeInMonths'] = train_df['AgeuponOutcome'].apply(parse_age_in_months)
test_df['AgeInMonths'] = test_df['AgeuponOutcome'].apply(parse_age_in_months)

In [6]:
for df in [train_df, test_df]:
    df['Name'] = df['Name'].fillna('No Name')
    df['SexuponOutcome'] = df['SexuponOutcome'].fillna('Unknown')

for fold in range(5):
    train_fold = train_df[train_df['fold'] != fold]
    val_idx = train_df[train_df['fold'] == fold].index

    type_medians = train_fold.groupby('AnimalType')['AgeInMonths'].median()

    for animal_type, median_val in type_medians.items():
        mask = (train_df.index.isin(val_idx)) & (train_df['AnimalType'] == animal_type) & (train_df['AgeInMonths'].isnull())
        train_df.loc[mask, 'AgeInMonths'] = median_val

overall_medians = train_df.groupby('AnimalType')['AgeInMonths'].median()
for animal_type, median_val in overall_medians.items():
    test_df.loc[(test_df['AnimalType'] == animal_type) & (test_df['AgeInMonths'].isnull()), 'AgeInMonths'] = median_val

In [7]:
print("Maradt hiányzó érték a train_df-ben?")
print(train_df[['Name', 'SexuponOutcome', 'AgeInMonths']].isnull().sum())

print("\nMaradt hiányzó érték a test_df-ben?")
print(test_df[['Name', 'SexuponOutcome', 'AgeInMonths']].isnull().sum())

train_df.to_csv('train_cleaned.csv', index=False)
test_df.to_csv('test_cleaned.csv', index=False)
print("\nA 'train_cleaned.csv' és 'test_cleaned.csv' sikeresen elmentve!")

Maradt hiányzó érték a train_df-ben?
Name              0
SexuponOutcome    0
AgeInMonths       0
dtype: int64

Maradt hiányzó érték a test_df-ben?
Name              0
SexuponOutcome    0
AgeInMonths       0
dtype: int64

A 'train_cleaned.csv' és 'test_cleaned.csv' sikeresen elmentve!
